# 3. Review the segments and write the manifests

1. **Review** of what 2_segment found in this run: the contact sheet of every broadcast, to drop
   segments that are not interpretation, or cut them where the face search was fooled. A human
   writes the outcome to `decisions.json`, as frames of the broadcast, so that it holds for ranges
   found a frame or two differently.
2. **Manifests**: the segments of the release (`common.found`), as users download them, minus the
   decisions, in `work/release/manifests.json` (4_labels adds the splits).

In 'replay' mode there is nothing to review: the manifests are built from `frozen/` and `decisions.json`.

In [ ]:
from common import CHANNELS, CROP, DECISIONS, FPS, MODE, WORK, digi24_era, found, read, write

## 1. Review

2_segment's contact sheet of every broadcast with an interpreter (`work/sheets/`): the bounds of
its segments and frames across them, and for a broadcast done again the stretches it gained or
lost. Broadcasts with short segments come first: a face in the box that is not the interpreter's
(one in a crowd) can make one. In `decisions.json`, per channel, `dropped_segments` lists the
segments that are not interpretation and `cuts` the frames to keep of those the face search
fooled at their start or end: per video or day, [start, end) frames as users download them.

Before them, the broadcasts in which an interpreter was expected but no known face was found
(`work/unrecognized.json`): one that shows an interpreter not in `known_people/` needs their
picture there and their name in `common.INTERPRETERS` (Digi24: a reference segment in
`decisions.json`), then 2_segment again with the broadcast in `REDO`.

In [ ]:
if MODE != 'replay':
    if (WORK / 'unrecognized.json').exists():
        for channel, keys in read(WORK / 'unrecognized.json').items():
            for key in keys:
                print('NO KNOWN INTERPRETER:', channel, key, WORK / 'sheets' / channel / f'{key}.jpg')
    for channel in CHANNELS:
        run = {k: v for k, v in read(WORK / 'segments.json')[channel].items() if v}
        for key in sorted(run, key=lambda k: min(e - s for s, e in run[k]['ranges'])):
            print(channel, key, [round((e - s) / FPS) for s, e in run[key]['ranges']], 's',
                  WORK / 'sheets' / channel / f'{key}.jpg')

## 2. Manifests

One item per broadcast with a known interpreter that the archive still serves, with its
segments minus the dropped ones and cut as decided; the interpreter box (`crop`) is the one of
the broadcast's height. Items are listed in the order the broadcasts were found.

In [ ]:
decisions = read(DECISIONS)
items = {c: [] for c in CHANNELS}
release = found()


def overlap(a, b):
    return max(0, min(a[1], b[1]) - max(a[0], b[0]))


def kept(channel, key, segments):
    """The segments minus those mostly inside a dropped span, each cut to the spans of `cuts` it overlaps; one that
    overlaps several is split, the parts after its first numbered after the broadcast's last segment."""
    cuts, dropped = decisions[channel]['cuts'].get(key, []), decisions[channel]['dropped_segments'].get(key, [])
    out, n = [], max((s['index'] + 1 for s in segments), default=0)
    for s in segments:
        span = (s['start'], s['end'])
        if any(overlap(span, d) * 2 > span[1] - span[0] for d in dropped):
            continue
        parts = [(max(span[0], a), min(span[1], b)) for a, b in cuts if overlap(span, (a, b))] or [span]
        for k, (a, b) in enumerate(parts):
            index, n = (s['index'], n) if k == 0 else (n, n + 1)
            out.append({'index': index, 'start': a, 'end': b})
    return sorted(out, key=lambda s: s['start'])


for channel in CHANNELS:
    for key, record in release[channel].items():
        if record is None or record['interpreter'] is None or 'unavailable' in record \
                or key in decisions[channel].get('dropped_days', {}):
            continue
        name = key.split('/')[0] if channel == 'ProTV' else key
        shifted = [{'index': i, 'start': s + d, 'end': e + d}     # not checked inside: as found
                   for i, ((s, e), d) in enumerate(zip(record['ranges'], record['shift']))]
        segments, height = kept(channel, name, record.get('segments', shifted)), record['source_height']
        if channel == 'PrimaTV':
            items[channel].append({'id': key, 'url': record['source'], 'interpreter': record['interpreter'],
                                   'source_height': height, 'crop': CROP[channel][height], 'segments': segments})
        elif channel == 'ProTV':
            items[channel].append({'date': name, 'url': record['source'], 'interpreter': record['interpreter'],
                                   'crop': CROP[channel][height], 'source_height': height, 'segments': segments})
        else:
            items[channel].append({'date': name, 'url': record['source'], 'video_src': record['video_src'],
                                   'interpreter': record['interpreter'], 'crop': CROP[channel][digi24_era(name)],
                                   'source_height': height, 'segments': segments})
assert len({it['date'] for it in items['ProTV']}) == len(items['ProTV']), 'one ProTV broadcast per day'

write(WORK / 'release' / 'manifests.json', items)
print({c: (len(items[c]), sum(len(it['segments']) for it in items[c])) for c in CHANNELS})